In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir(r"E:\retailpulse")

df = pd.read_csv("data/processed/master.csv",
                 parse_dates=["order_date"])

# Snapshot date = 1 day after last order
snapshot_date = df["order_date"].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date}")

# Build RFM at customer level
rfm = df.groupby("customer_id_unique").agg(
    recency   = ("order_date",
                 lambda x: (snapshot_date - x.max()).days),
    frequency = ("order_id", "nunique"),
    monetary  = ("revenue", "sum")
).reset_index()

print(f"\nRFM shape: {rfm.shape}")
print(f"\nRFM Summary:")
print(rfm[["recency","frequency","monetary"]].describe().round(2))

Snapshot date: 2018-08-30 15:00:37

RFM shape: (93350, 4)

RFM Summary:
        recency  frequency  monetary
count  93350.00   93350.00  93350.00
mean     237.95       1.03    165.17
std      152.59       0.21    226.30
min        1.00       1.00      9.59
25%      114.00       1.00     63.01
50%      219.00       1.00    107.78
75%      346.00       1.00    182.50
max      714.00      15.00  13664.08


In [2]:
# Score each R, F, M on a scale of 1-4
# Recency: lower days = better = higher score
# Frequency & Monetary: higher = better = higher score

rfm["R_score"] = pd.qcut(rfm["recency"],
                          q=4,
                          labels=[4,3,2,1])  # reversed
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"),
                          q=4,
                          labels=[1,2,3,4])
rfm["M_score"] = pd.qcut(rfm["monetary"],
                          q=4,
                          labels=[1,2,3,4])

rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

# Combined RFM score
rfm["RFM_score"] = (rfm["R_score"].astype(str) +
                    rfm["F_score"].astype(str) +
                    rfm["M_score"].astype(str))

rfm["Total_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

print("RFM Scores Sample:")
print(rfm.head(10).to_string(index=False))

RFM Scores Sample:
              customer_id_unique  recency  frequency  monetary  R_score  F_score  M_score RFM_score  Total_score
0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90        4        1        3       413            8
0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19        3        1        1       311            5
0000f46a3911fa3c0805444483337064      537          1     86.22        1        1        2       112            4
0000f6ccb0745a6a4b88665a16c9f078      321          1     43.62        2        1        1       211            4
0004aac84e0df4da2b147fca70cf8255      288          1    196.89        2        1        4       214            7
0004bd2a26a76fe21f786e4fbd80607f      146          1    166.98        3        1        3       313            7
00050ab1314c0e55a6ca13cf7181fecf      132          1     35.38        3        1        1       311            5
00053a61a98854899e70ed204dd4bafe      183          1    419.18        3      

In [3]:
# Assign segments based on RFM scores
def assign_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]
    total = row["Total_score"]

    if r >= 4 and f >= 3:
        return "Champions"
    elif r >= 3 and f >= 2:
        return "Loyal Customers"
    elif r >= 3 and f == 1:
        return "Potential Loyalists"
    elif r == 2 and f >= 2:
        return "At Risk"
    elif r == 1 and f >= 2:
        return "Cant Lose Them"
    elif r <= 2 and f == 1 and m >= 3:
        return "Big Spenders"
    elif r == 1 and f == 1:
        return "Lost"
    else:
        return "Hibernating"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

print("📊 Customer Segments:")
seg_counts = rfm["segment"].value_counts().reset_index()
seg_counts.columns = ["segment","count"]
seg_counts["percentage"] = (seg_counts["count"] /
                             len(rfm) * 100).round(1)
print(seg_counts.to_string(index=False))

📊 Customer Segments:
            segment  count  percentage
    Loyal Customers  23458        25.1
            At Risk  17436        18.7
     Cant Lose Them  17408        18.6
          Champions  11710        12.5
Potential Loyalists  11673        12.5
       Big Spenders   5654         6.1
               Lost   3044         3.3
        Hibernating   2967         3.2


In [4]:
# Pie chart of segments
fig = px.pie(seg_counts,
             values="count",
             names="segment",
             title="Customer Segments Distribution",
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.show()

# Scatter plot: Recency vs Monetary colored by segment
fig2 = px.scatter(rfm.sample(5000, random_state=42),
                  x="recency",
                  y="monetary",
                  color="segment",
                  title="Customer Segments — Recency vs Monetary Value",
                  labels={"recency":"Days Since Last Purchase",
                          "monetary":"Total Spend (BRL)"},
                  opacity=0.6)
fig2.show()

# Segment summary table
seg_summary = rfm.groupby("segment").agg(
    count     = ("customer_id_unique", "count"),
    avg_recency   = ("recency",   "mean"),
    avg_frequency = ("frequency", "mean"),
    avg_monetary  = ("monetary",  "mean"),
    total_revenue = ("monetary",  "sum")
).round(2).reset_index()

print("\n📋 Segment Summary:")
print(seg_summary.to_string(index=False))


📋 Segment Summary:
            segment  count  avg_recency  avg_frequency  avg_monetary  total_revenue
            At Risk  17436       277.38           1.04        163.09     2843609.79
       Big Spenders   5654       366.73           1.00        267.64     1513214.75
     Cant Lose Them  17408       451.20           1.04        164.88     2870198.76
          Champions  11710        57.40           1.08        177.13     2074175.14
        Hibernating   2967       276.99           1.00         63.80      189293.60
               Lost   3044       451.01           1.00         63.70      193908.47
    Loyal Customers  23458       139.22           1.04        164.55     3860015.17
Potential Loyalists  11673       112.71           1.00        160.54     1873979.15


In [5]:
rfm.to_csv("data/processed/rfm.csv", index=False)
seg_summary.to_csv("data/processed/segment_summary.csv", index=False)

print("✅ RFM saved!")
print(f"\n🎯 Most valuable segment: "
      f"{seg_summary.loc[seg_summary['total_revenue'].idxmax(),'segment']}")
print(f"⚠️  Largest at-risk group: "
      f"{seg_counts.iloc[0]['segment']} "
      f"({seg_counts.iloc[0]['percentage']}%)")

✅ RFM saved!

🎯 Most valuable segment: Loyal Customers
⚠️  Largest at-risk group: Loyal Customers (25.1%)
